# RBI/SEBI Regulation Copilot — built in 8 steps

**The problem.** RBI and SEBI are India's financial regulators. Their rules live in long PDFs ("Master Directions") — the KYC one alone is 100+ pages. Finding one rule means searching the PDF by hand, and a normal chatbot answers from memory with no source.

**What we build.** Type a question → get the answer **from the official PDF**, with the page number. If the PDFs don't cover it, it says so instead of guessing.

```
 1 Download PDFs → 2 Read pages → 3 Cut into chunks → 4 Turn chunks into numbers
                                                              ↓
 8 Agent (optional) ← 7 Test it ← 6 Answer with sources ← 5 Search for the best chunks
```

Steps 1–4 prepare the library once. Steps 5–6 run for every question. This idea is called **RAG — Retrieval-Augmented Generation**: *look it up first, then answer.*

**Run it:** Runtime → Run all. No API key needed. (Optional: add `ANTHROPIC_API_KEY` in the 🔑 Secrets panel for AI-written answers and step 8.)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install -q pymupdf rank_bm25 sentence-transformers anthropic
import os, re, json, html
import numpy as np
import pandas as pd
import requests

FOLDER = "/content/drive/MyDrive/regcopilot"     # everything is saved here
PDFS = f"{FOLDER}/pdfs"
os.makedirs(PDFS, exist_ok=True)

---
## Step 1 — Download the rulebooks
🎯 **What:** get RBI's Master Directions as PDFs.
💡 **How:** RBI has one web page listing them all. We find each topic on that page and download the PDF next to it.
👀 **Look at:** the list of saved files at the end.

In [ ]:
RBI_PAGE = "https://www.rbi.org.in/Scripts/BS_ViewMasterDirections.aspx"
DOCUMENTS = {                                   # topic on RBI's page -> name we save it as
    "Know Your Customer": "RBI_KYC_Master_Direction.pdf",
    "Prepaid Payment Instruments": "RBI_PPI_Master_Direction.pdf",
    "Fraud Risk Management": "RBI_Fraud_Risk_Management.pdf",
    "Payment Aggregator": "RBI_Payment_Aggregators.pdf",
}
browser = requests.Session()                    # acts like one browser tab
browser.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/126.0", "Referer": RBI_PAGE})
page_html = browser.get(RBI_PAGE, timeout=60).text
print(len(page_html), "characters on RBI's page")

In [ ]:
def find_pdf(topic):
    """Find the topic on RBI's page and return (pdf_link, title)."""
    found = []
    for m in re.finditer(re.escape(topic), page_html, flags=re.I):
        start = max(page_html.rfind(">", max(0, m.start() - 200), m.start()) + 1, m.start() - 200)
        raw = re.split(r"<a\b|</td>|</tr>|<br", page_html[start: m.start() + 300], flags=re.I)[0]
        title = re.sub(r"<[^>]+>|\s+", " ", html.unescape(raw)).strip()           # text only, no HTML
        if any(w in title.lower() for w in ["ombudsman", "amendment", "repeal"]):  # not the main rulebook
            continue
        link = re.search(r"""https?://rbidocs\.rbi\.org\.in/[^"'\s>]+?\.pdf""", page_html[m.start(): m.start() + 3000], flags=re.I)
        if link:
            found.append((link.group(0), title))
    # prefer Master Direction files (named like 169MD.PDF) over notifications about the same topic
    for link, title in found:
        if re.search(r"/\d+MD[^/]*\.pdf$", link, flags=re.I):
            return link, title
    return found[0] if found else None

In [ ]:
for topic, filename in DOCUMENTS.items():
    if os.path.exists(f"{PDFS}/{filename}"):
        print("already have", filename); continue
    result = find_pdf(topic)
    if result is None:
        print("NOT FOUND:", topic); continue
    link, title = result
    pdf = browser.get(link, timeout=120).content
    if pdf[:4] == b"%PDF":                       # a real PDF always starts with %PDF
        open(f"{PDFS}/{filename}", "wb").write(pdf)
        print(f"saved {filename} ({len(pdf)//1024} KB) - {title[:60]}")

sorted(os.listdir(PDFS))

---
## Step 2 — Read the PDFs, page by page
🎯 **What:** pull the text out of every page into a table.
💡 **Why keep the page number?** So every answer can say *"KYC Direction, page 41"*.
👀 **Look at:** `empty_pages` — pages with almost no words are scanned images that can't be searched.

In [ ]:
import pymupdf

rows = []
for filename in sorted(os.listdir(PDFS)):
    regulator = filename.split("_")[0]                          # "RBI_KYC_..." -> "RBI"
    title = filename[len(regulator) + 1:-4].replace("_", " ")   # -> "KYC Master Direction"
    with pymupdf.open(f"{PDFS}/{filename}") as pdf:
        for number, page in enumerate(pdf, start=1):
            rows.append({"file": filename, "regulator": regulator, "title": title, "page": number, "text": page.get_text()})

data = pd.DataFrame(rows)                                       # one row per page
data["words"] = data["text"].str.split().str.len()
data.groupby("title").agg(pages=("page", "count"), words=("words", "sum"), empty_pages=("words", lambda w: (w < 20).sum()))

---
## Step 3 — Cut the pages into small pieces ("chunks")
🎯 **What:** cut every page into pieces of ~300 words; neighbouring pieces share 50 words.
💡 **Why:** we can't send a 100-page rulebook to the AI for every question. We'll find the 5 best pieces and send only those. The overlap means a rule that sits across a cut still appears whole in one piece.
👀 **Look at:** one example chunk — it should read like a paragraph of the rulebook.

In [ ]:
def clean(text):
    text = text.replace("\u00a0", " ")             # odd invisible spaces
    text = re.sub(r"-\n(\w)", r"\1", text)         # "regu-\nlation" -> "regulation"
    return re.sub(r"\s+", " ", text).strip()        # many spaces / new lines -> one space

def make_chunks(text, size=300, overlap=50):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
        start += size - overlap                      # move on 250 words, so 50 repeat
    return chunks

print(len(make_chunks("w " * 700)), "chunks from 700 words (should be 3)")

In [ ]:
chunk_rows = []
for _, row in data.iterrows():
    for i, piece in enumerate(make_chunks(clean(row["text"]))):
        chunk_rows.append({"chunk_id": f"{row['file'][:-4]}_p{row['page']}_c{i}",
                           "title": row["title"], "page": row["page"], "text": piece})
chunks = pd.DataFrame(chunk_rows)
print(len(chunks), "chunks")
print(chunks.iloc[len(chunks) // 2]["text"][:500])

---
## Step 4 — Turn every chunk into numbers ("embeddings")
🎯 **What:** a free AI model reads each chunk and turns it into **384 numbers**.
💡 **Why:** texts with similar **meaning** get similar numbers — even with different words. So *"how often re-verify customers?"* can match *"periodic updation of KYC shall be carried out…"*.
👀 **Look at:** A vs B (same meaning, different words) should score clearly higher than A vs C.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")      # free, downloads once (~130 MB)

v = embedder.encode(["KYC must be updated periodically for high-risk customers",          # A
                     "Customer identity details need re-verification every two years",   # B: same meaning
                     "The payment aggregator must have a minimum net worth"],            # C: other topic
                    normalize_embeddings=True)
print(f"A vs B: {v[0] @ v[1]:.2f}   (same meaning)")
print(f"A vs C: {v[0] @ v[2]:.2f}   (different topic)")

In [ ]:
vectors = embedder.encode(chunks["text"].tolist(), normalize_embeddings=True, batch_size=32, show_progress_bar=True)
print(vectors.shape, "-> one row of 384 numbers per chunk")

---
## Step 5 — Search for the best chunks
Three searches, combined:
1. **Meaning search** — compare the question's numbers with every chunk's numbers. Good with different wording.
2. **Keyword search (BM25)** — chunks containing the question's words (rare words count more). Good with exact terms like *"Rs.10,000"*.
3. **Re-ranker** — a second model reads the question **and** each candidate together and scores how well it answers. It only checks the top 20, because it's slower.

👀 **Look at:** the comparison — the FINAL list should have the exact rule at the top.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())

keyword_index = BM25Okapi([words(t) for t in chunks["text"]])
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")     # free (~90 MB)

def meaning_search(question, k=20):
    q = embedder.encode(["Represent this sentence for searching relevant passages: " + question], normalize_embeddings=True)[0]
    scores = vectors @ q
    best = np.argsort(-scores)[:k]
    return chunks.iloc[best].assign(score=scores[best])

def keyword_search(question, k=20):
    scores = keyword_index.get_scores(words(question))
    best = np.argsort(-scores)[:k]
    return chunks.iloc[best].assign(score=scores[best])

In [ ]:
def search(question, top_k=5):
    # combine the two lists by RANK (Reciprocal Rank Fusion): near the top of both lists = best
    points = {}
    for ranking in (meaning_search(question), keyword_search(question)):
        for rank, cid in enumerate(ranking["chunk_id"], start=1):
            points[cid] = points.get(cid, 0) + 1 / (60 + rank)
    top20 = sorted(points, key=points.get, reverse=True)[:20]
    candidates = chunks.set_index("chunk_id").loc[top20].reset_index()
    # the re-ranker reads question + chunk together and scores each one
    candidates["score"] = reranker.predict([(question, t) for t in candidates["text"]])
    return candidates.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)

q = "What is the maximum amount that can be loaded in a small PPI?"
for name, found in [("meaning only", meaning_search(q, 3)), ("keyword only", keyword_search(q, 3)), ("FINAL", search(q, 3))]:
    print(f"\n{name}:")
    for r in found.itertuples():
        print(f"  {r.title}, p.{r.page} | {r.text[:90]}...")

---
## Step 6 — Answer, only from the chunks found
This is the **RAG** moment: **R**etrieve (step 5) → **A**ugment (number the chunks [1]–[5]) → **G**enerate (write the answer citing [1], [2]).

Two ways to write the answer — so it **always works**, even in an interview with no API key:
- **Claude** writes it, if a key is saved in 🔑 Secrets.
- **Quote mode** (no AI): the best sentences from the chunks, word for word.

🛡️ **Safety check first:** if even the best chunk scores below 0, the PDFs don't cover the question → *"I couldn't find this…"*. (Real questions score above 4; *"current repo rate"* scores about −10.)

In [ ]:
NOT_FOUND = "I couldn't find this in the provided RBI/SEBI documents."
RULES = f"""Answer questions about Indian financial regulations using ONLY the numbered sources.
End every factual sentence with its source number, e.g. [2]. Quote numbers exactly.
If the sources don't contain the answer, say exactly: "{NOT_FOUND}" Keep it to 2-5 sentences."""

def numbered_sources(found):
    return "\n\n".join(f"[{i}] {r.title}, page {r.page}\n{r.text}" for i, r in enumerate(found.itertuples(), start=1))

def quote_answer(question, found):
    options = [(i, s.strip()) for i, r in enumerate(found.itertuples(), start=1)
               for s in re.split(r"(?<=[.;:])\s+", r.text) if 8 <= len(s.split()) <= 80]
    scores = reranker.predict([(question, s) for _, s in options])
    best = sorted(zip(scores, options), key=lambda x: x[0], reverse=True)[:2]
    keep = [f"{s} [{i}]" for score, (i, s) in best if score >= 0]
    return " ".join(keep) or NOT_FOUND

In [ ]:
from google.colab import userdata
import anthropic

claude = None
try:
    claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    print("Claude key found - answers will be written by Claude")
except Exception:
    print("No key - answers will quote the regulation word for word")

def ask(question):
    found = search(question)                                    # R
    if found["score"].iloc[0] < 0:                              # safety check
        return NOT_FOUND, "not in the documents", found
    if claude:
        try:
            reply = claude.messages.create(model="claude-haiku-5-5", max_tokens=400, system=RULES,
                messages=[{"role": "user", "content": f"Sources:\n\n{numbered_sources(found)}\n\nQuestion: {question}"}])   # A
            return reply.content[0].text, "Claude", found       # G
        except Exception as e:
            print("Claude unavailable:", type(e).__name__)
    return quote_answer(question, found), "quote (no AI)", found

In [ ]:
for q in ["How often must KYC be updated for high risk customers?",
          "What is the maximum amount that can be loaded in a small PPI in a month?",
          "What is the current repo rate?"]:                  # not in our PDFs: must refuse
    answer, method, found = ask(q)
    print(f"\nQ: {q}\nA ({method}): {answer}")
    for n in sorted({int(x) for x in re.findall(r"\[(\d+)\]", answer)}):
        print(f"   [{n}] {found.iloc[n-1]['title']}, p.{found.iloc[n-1]['page']}")

---
## Step 7 — Test it on questions with known answers
🎯 **What:** 8 questions whose answers we know, and 3 it must refuse. We check each one automatically.
💡 **Why:** "it looks right" isn't proof. A score is. (The GitHub version runs this every week and refuses to publish if a score drops.)

In [ ]:
tests = [
    ("How often must KYC be updated for high-risk customers?", "two years"),
    ("How often is periodic updation of KYC required for medium risk customers?", "eight years"),
    ("How often is periodic updation of KYC required for low risk customers?", "10 years"),
    ("What is the maximum amount that can be loaded in a small PPI in a month?", "10,000"),
    ("What is the total amount that can be loaded in a small PPI during a financial year?", "1,20,000"),
    ("What is the cash withdrawal limit per transaction for non-bank PPIs?", "2,000"),
    ("Within how many days must a bank report theft, burglary or robbery to RBI?", "seven days"),
    ("How much time must a bank give a person to reply to a show cause notice before classifying a fraud?", "21 days"),
    ("What is the current repo rate?", None),                  # None = must refuse
    ("What is the GST rate on restaurant bills?", None),
    ("Who won the IPL final in 2024?", None),
]
loose = lambda t: re.sub(r"[\s,/\-]", "", str(t).lower())     # "Rs.1,20,000/-" contains "1,20,000"

results = []
for question, expected in tests:
    answer, method, found = ask(question)
    passed = (method == "not in the documents") if expected is None else loose(expected) in loose(answer)
    results.append({"question": question, "expected": expected or "refuse", "passed": passed})
results = pd.DataFrame(results)
print(f"Passed {results['passed'].sum()} of {len(results)}")
results

---
## Step 8 — The agent: let the AI decide what to look up *(needs the Claude key)*
Step 6 searches **once**. An **agent** is an AI in a loop that can use **tools** and decide what to do next:

```
question → AI: "I'll search high-risk KYC" → we run search → AI reads it
         → AI: "now low-risk KYC"          → we run search → AI reads it → AI writes the answer
```

In [ ]:
TOOLS = [{"name": "search_rules", "description": "Search the RBI/SEBI documents. Returns 5 chunks with chunk_id, title, page, text.",
          "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}]

def run_agent(question, max_steps=6):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = claude.messages.create(model="claude-haiku-5-5", max_tokens=800, tools=TOOLS, messages=messages,
                                       system="Research RBI/SEBI rules using only your tools. Cite chunk_ids in [brackets]. Plan one search per part.")
        if reply.stop_reason != "tool_use":                       # ready to answer
            return " ".join(b.text for b in reply.content if b.type == "text")
        messages.append({"role": "assistant", "content": reply.content})
        results = []
        for block in reply.content:
            if block.type == "tool_use":
                print(f"  step {step}: search_rules('{block.input['query']}')")   # watch it think
                found = search(block.input["query"])[["chunk_id", "title", "page", "text"]].to_json(orient="records")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": found})
        messages.append({"role": "user", "content": results})
    return "Stopped: too many steps."

if claude:
    print(run_agent("Compare how often KYC must be updated for high, medium and low risk customers."))
else:
    print("Step 8 needs ANTHROPIC_API_KEY in the 🔑 Secrets panel.")

In [ ]:
drive.flush_and_unmount()       # always the LAST cell